# 3. Rendimiento emparejado en NVIDIA Jetson Orin Nano
Si los notebooks 01 y 02 ya se ejecutaron en esta placa y siguen disponibles sus runs, trazas e informe emparejado, dejar TRANSFER = None: basta volver a ejecutar este notebook. Si los modelos se entrenaron en otro equipo, indicar la ruta de su bundle privado. La nueva medición compara el throughput de un replay causal completo sobre los mismos objetivos físicos; no mide la latencia individual de un flujo vivo.


In [5]:
from pathlib import Path
import json
import platform
import sys
import pandas as pd
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT, validate_splits
from shared.reporting.edge_bundle import verify, install
assert Path.cwd().resolve() == ROOT
assert sys.platform.startswith('linux')
print(platform.platform(), platform.machine(), sys.version)
TRANSFER = None  # Para un bundle externo: Path('/ruta/privada/al/bundle')
LIGHTGBM_RUN = None  # Opcional: se comprobará contra el informe emparejado
LSTM_RUN = None      # Opcional: se comprobará contra el informe emparejado


Linux-5.15.148-tegra-aarch64-with-glibc2.35 aarch64 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]


## Selección y verificación
En modo directo se seleccionan los runs por model_version del informe emparejado del notebook 02. Pueden existir runs anteriores; no se borra ninguno. Si se indican rutas manuales, deben coincidir con el informe. En modo bundle se verifican e instalan archivos ignorados sin sobrescribir ninguno distinto. La nueva razón requiere el LSTM causal v2.


In [6]:
if TRANSFER is None:
    contract = validate_splits()
    quality = json.loads((ROOT / 'results' / 'metrics' / 'paired_test.json').read_text(encoding='utf-8'))
    assert quality['paired_test_rows'] == 31680
    assert all(quality[name]['data_manifest_id'] == contract['manifest_id'] for name in ('lightgbm', 'lstm'))
    def select_run(kind, approach, selected):
        version = quality[approach]['model_version']
        prefix = {'lightgbm_night_v1': 'lightgbm-night-context-v1-', 'lstm_causal_v1': 'lstm-causal-v1-', 'lstm_causal_v2': 'lstm-causal-v2-'}[kind]
        assert version.startswith(prefix), f'{approach}: versión de modelo inesperada: {version}'
        run_id = version[len(prefix):]
        assert run_id and Path(run_id).name == run_id, f'{approach}: ID de run inválido'
        expected = (ROOT / 'experiments' / 'runs' / kind / run_id).resolve()
        assert (expected / 'manifest.json').is_file(), f'{approach}: falta el run del informe: {expected}'
        if selected is not None:
            assert (ROOT / selected).resolve() == expected, f'{approach}: el run elegido no coincide con el informe emparejado'
        return expected
    lightgbm_run = select_run('lightgbm_night_v1', 'lightgbm', LIGHTGBM_RUN)
    lstm_run = select_run('lstm_causal_v2' if quality['lstm']['model_version'].startswith('lstm-causal-v2-') else 'lstm_causal_v1', 'lstm', LSTM_RUN)
else:
    plan = verify(TRANSFER)
    install(TRANSFER)
    assert validate_splits()['manifest_id'] == plan['data_manifest_id']
    runs = json.loads((TRANSFER / 'bundle_manifest.json').read_text(encoding='utf-8'))['runs']
    lightgbm_run, lstm_run = ROOT / runs['lightgbm'], ROOT / runs['lstm']
print(lightgbm_run, lstm_run, sep='\n')


/home/jetson/Desktop/greenhouse-online-anomaly-detection/experiments/runs/lightgbm_night_v1/20261105T093300Z
/home/jetson/Desktop/greenhouse-online-anomaly-detection/experiments/runs/lstm_causal_v2/20261105T093456Z


## Paridad de inferencia antes del cronómetro
Se cargan primero las trazas de referencia; luego se vuelve a inferir y se comprueba que las 31.680 predicciones no cambian. En modo directo, las referencias proceden del notebook 01 y del informe validado por el notebook 02. El benchmark vuelve a comprobar la paridad durante cada pasada medida.


In [7]:
from approaches.lightgbm.night_context import export_test as export_lightgbm
from approaches.lstm.adapter import export_test as export_lstm
reference_root = ROOT / ('results/traces' if TRANSFER is None else 'experiments/runs/edge_reference')
expected = {name: pd.read_parquet(reference_root / f'{name}_test.parquet') for name in ('lightgbm_night', 'lstm')}
assert len(expected['lightgbm_night']) == len(expected['lstm']) == 31680
assert set(expected['lightgbm_night'].model_version) == {f'lightgbm-night-context-v1-{lightgbm_run.name}'}
assert set(expected['lstm'].model_version) == {f'lstm-causal-{lstm_run.parent.name[-2:]}-{lstm_run.name}'}
if TRANSFER is None:
    assert quality['lightgbm']['model_version'] == expected['lightgbm_night'].model_version.iloc[0]
    assert quality['lstm']['model_version'] == expected['lstm'].model_version.iloc[0]
for name, produced in [('lightgbm_night', export_lightgbm(lightgbm_run)[0]), ('lstm', export_lstm(lstm_run)[0])]:
    actual = pd.read_parquet(produced)
    columns = ['timestamp', 'ground_truth_type', 'binary_prediction', 'stage_2_prediction', 'final_prediction']
    assert actual[columns].equals(expected[name][columns]), f'{name}: la inferencia cambió'
    print(name, len(actual), 'objetivos idénticos')
import gc
import tensorflow as tf
tf.keras.backend.clear_session()
gc.collect()


lightgbm_night 31680 objetivos idénticos
lstm 31680 objetivos idénticos


0

## Replay causal completo y razón de velocidad
La medición usa los mismos timestamps y una decisión final por objetivo. LightGBM incluye máscara de ausencias, reloj nominal, extracción causal, Model 1, persistencia, reglas y Model 2 condicional. LSTM incluye imputación causal, ingeniería y escalado de variables de test, ensamblaje de ventanas, Model 1 y Model 2 condicional. La carga de archivos y modelos, y el ajuste de escaladores con train, quedan fuera de ambos cronómetros. Se hace una pasada completa de calentamiento y cinco repeticiones. LightGBM se mide antes y después del LSTM para detectar deriva térmica.

Por defecto se usan los 31.680 objetivos del test. Para comprobar el recorrido antes de la medición larga, cambiar temporalmente ROWS a 2000. El informe solo se guarda si las predicciones coinciden con las trazas y los tiempos son suficientemente estables. La razón resultante describe throughput del replay completo; la preparación LSTM se vectoriza sobre el bloque de test, así que no representa latencia individual de streaming.


In [8]:
from shared.benchmarking.full_pipeline import run_pair
ROWS = 31680  # Para una prueba corta: 2000; volver a 31680 para la cifra principal
REPEATS = 5
quality_path = ROOT / ('results/metrics/paired_test.json' if TRANSFER is None else 'experiments/runs/edge_reference/paired_test.json')
report_path = run_pair(lightgbm_run, lstm_run, rows=ROWS, repeats=REPEATS, quality_path=quality_path)
report = json.loads(report_path.read_text(encoding='utf-8'))
assert report['rows_per_pass'] == ROWS and report['lightgbm']['parity_with_test_trace'] and report['lstm']['parity_with_test_trace']
summary = pd.DataFrame([
    {'enfoque': name, 'objetivos_por_pasada': ROWS, 'repeticiones': REPEATS,
     'ms_por_objetivo': report[name]['mean_ms_per_target'],
     'objetivos_por_segundo': report[name]['throughput_targets_per_second'],
     'invocaciones_model2_por_pasada': report[name]['stage2_invocations_per_pass']}
    for name in ('lightgbm', 'lstm')
])
display(summary)
print(f"Razón de throughput LightGBM/LSTM: {report['speed_ratio_lightgbm_over_lstm_full_replay']:.3f}x")
print('Informe completo:', report_path)


ImportError: cannot import name 'transform_test_for_benchmark' from 'approaches.lstm.causal_data' (/home/jetson/Desktop/greenhouse-online-anomaly-detection/approaches/lstm/causal_data.py)

El informe y las dos mediciones LightGBM de control quedan en results/reports/, ignorados por Git. El JSON registra dispositivo, versiones, modo de potencia si está disponible, hashes de modelos, número de objetivos, repeticiones, paridad y estabilidad. Registrar también temperatura y frecuencias durante la sesión para interpretar diferencias. Publicar la razón como throughput de replay causal completo, con el alcance indicado en el informe; no como latencia de una decisión individual en vivo.
